In [10]:
import geopandas as gpd
import pandas as pd
import folium
import os

In [11]:
run_stats = gpd.read_file('data/processed/run_stats_valid_runs.geojson')
pistes_gdf = gpd.read_file('data/vicheres_pistes.geojson')

In [14]:

# Assumes:
# - run_stats has: piste_id, duration_s, name, plot_color
# - pistes_gdf has: piste_id, geometry (and optionally name/plot_color)

def fmt_time(sec):
    sec = int(round(sec))
    m, s = divmod(sec, 60)
    h, m = divmod(m, 60)
    return f"{h}:{m:02d}:{s:02d}" if h else f"{m}:{s:02d}"

# 1) Build per-piste summary (run count + top 3 times)
summary = (
    run_stats.groupby("piste_id")
    .apply(lambda g: pd.Series({
        "name": g["name"].dropna().iloc[0] if g["name"].notna().any() else f"Piste {g.name}",
        "plot_color": g["plot_color"].dropna().iloc[0] if g["plot_color"].notna().any() else "#1f77b4",
        "run_count": len(g),
        "fastest_1": fmt_time(g["duration_s"].nsmallest(1).iloc[0]) if len(g) >= 1 else None,
        "fastest_2": fmt_time(g["duration_s"].nsmallest(2).iloc[-1]) if len(g) >= 2 else None,
        "fastest_3": fmt_time(g["duration_s"].nsmallest(3).iloc[-1]) if len(g) >= 3 else None,
    }))
    .reset_index()
)

# 2) One geometry per piste (from pistes_gdf), join summary
pistes_one = pistes_gdf.dissolve(by="id", as_index=False)
map_gdf = pistes_one.merge(summary, left_on="id", right_on="piste_id", how="inner")
# after merge, before folium.GeoJson(...)
map_gdf["name"] = map_gdf["name_y"].fillna(map_gdf["name_x"])  # pick preferred source

# 3) Web map wants WGS84 lat/lon
map_gdf = map_gdf.to_crs(4326)

# 4) Create map centered on resort
center = map_gdf.union_all().centroid
m = folium.Map(location=[center.y, center.x], zoom_start=13, tiles=None)

# Basemap options
folium.TileLayer(
    tiles="OpenStreetMap",
    name="OpenStreetMap",
    control=True,
).add_to(m)

stadia_key = os.getenv("STADIA_API_KEY")
stadia_url = "https://tiles.stadiamaps.com/tiles/stamen_terrain/{z}/{x}/{y}.png"
if stadia_key:
    stadia_url = f"{stadia_url}?api_key={stadia_key}"

folium.TileLayer(
    tiles=stadia_url,
    attr='&copy; <a href="https://stadiamaps.com/">Stadia Maps</a> &copy; <a href="https://stamen.com/">Stamen Design</a> &copy; <a href="https://openmaptiles.org/">OpenMapTiles</a> &copy; <a href="https://www.openstreetmap.org/copyright">OpenStreetMap</a> contributors',
    name="Stadia Terrain",
    max_zoom=20,
    control=True,
).add_to(m)

# 5) Clickable pistes with popup
folium.GeoJson(
    map_gdf,
    style_function=lambda feat: {
        "color": feat["properties"]["plot_color"] or "#1f77b4",
        "weight": 4,
        "opacity": 0.9
    },
    highlight_function=lambda feat: {"weight": 6, "opacity": 1.0},
    tooltip=folium.GeoJsonTooltip(fields=["name"], aliases=["Piste"]),
    popup=folium.GeoJsonPopup(
        fields=["name", "run_count", "fastest_1", "fastest_2", "fastest_3"],
        aliases=["Piste", "Runs", "Fastest 1", "Fastest 2", "Fastest 3"],
        localize=True,
        labels=True
    )
).add_to(m)

# folium.TileLayer(
#     tiles="https://tiles.stadiamaps.com/tiles/stamen_terrain/{z}/{x}/{y}{r}.{ext}",
#     attr='Stadia Maps',
#     name="Terrain"
# ).add_to(m)

# # Optional labels overlay for readability
# folium.TileLayer(
#     tiles="https://services.arcgisonline.com/ArcGIS/rest/services/Reference/World_Boundaries_and_Places/MapServer/tile/{z}/{y}/{x}",
#     attr="Labels © Esri",
#     name="Labels",
#     overlay=True
# ).add_to(m)

folium.LayerControl().add_to(m)

m.save('docs/index.html')

m